# Resource estimation for simulating a 2D Ising model Hamiltonian

In this notebook we demonstrate how to estimate the resources for quantum dynamics, specifically the simulation of an Ising model Hamiltonian on an
$N \times N$ 2D lattice using a fourth-order Trotter Suzuki product formula assuming a 2D qubit architecture with nearest-neighbor connectivity.

We also showcase the impact of lattice-aware term grouping on the resource estimates of the Trotterized dynamics. By exploiting the lattice structure of models to organize their Hamiltonian terms into commuting and parallel layers prior to Trotterization, the corresponding circuits become cheaper to run fault-tolerantly (see the final figure in this notebook).

We use **qdk-chemistry** for all domain-specific functionality (model
Hamiltonians, Trotter decomposition, circuit construction) and
**qdk.qre** for quantum resource estimation.

## Background: 2D Ising model

The Ising model is a mathematical model of ferromagnetism in a lattice (in our case a 2D square lattice) with two kinds of terms in the Hamiltonian: (i) an interaction term between adjacent sites and (ii) an external magnetic field acting at each site. For our purposes, we consider a simplified version of the model where the interaction terms have the same strength and the external field strength is the same at each site.
Formally, the Ising model Hamiltonian on an $N \times N$ lattice we consider is formulated as:

$$
H = \underbrace{J \sum_{\left<i, j\right>} Z_i Z_j}_{B} + \underbrace{h \sum_j X_j}_{A}
$$
where $J$ is the interaction strength, $h$ is external field strength.

## Creating the Ising model

We use `qdk_chemistry` to define the lattice geometry and build the
Hamiltonian. `LatticeGraph.square` creates a 2D square lattice and
`create_ising_hamiltonian` produces the corresponding `QubitHamiltonian`.

By default `create_ising_hamiltonian` consults the lattice's optimal
**4-color edge coloring** for a 2D square lattice and stores the resulting
group-and-layer structure on `QubitHamiltonian.term_partition`. Edges of
the same color share no vertices, so the corresponding Pauli exponentials
can be applied in parallel within each color. The Trotter builder reads
`term_partition` automatically and uses it to reduce the number of
sequential layers per step.

In [ ]:
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import LatticeGraph
from qdk_chemistry.utils.model_hamiltonians import create_ising_hamiltonian

# Lattice dimensions
nx, ny = 10, 10

# Build the lattice and qubit Hamiltonian.  With ``include_term_groups=True``
# (the default), ``create_ising_hamiltonian`` consults the lattice's optimal
# edge coloring and stores the resulting layered group structure on
# ``QubitHamiltonian.term_partition``.  Downstream consumers (for example the
# Trotter builder) read this partition directly — no manual geometry
# boilerplate is required at the call site.
lattice = LatticeGraph.square(nx, ny)
hamiltonian = create_ising_hamiltonian(lattice, j=1.0, h=0.5)

partition = hamiltonian.term_partition
total_layers = sum(len(group) for group in partition.groups)
coloring = lattice.edge_coloring
ncolors = len(set(coloring.values())) if coloring else 0

print(f"Ising model on {lattice.num_sites}-site lattice "
      f"with {lattice.num_edges} edges")
print(f"QubitHamiltonian: {len(hamiltonian.pauli_strings)} Pauli terms, "
      f"{hamiltonian.num_qubits} qubits")
print(f"Edge coloring: {ncolors} colors")
print(f"Term partition (strategy={partition.strategy!r}): "
      f"{partition.num_groups} groups, {total_layers} total layers")

## Trotter expansion

The time evolution $e^{-iHt}$ for the Hamiltonian is simulated with the fourth-order product formula so that any errors in simulation are sufficiently small. Essentially, this is done by simulating the evolution for small slices of time $\Delta$ and repeating this for `nSteps` $= t/\Delta$ to obtain the full time evolution. The Trotter-Suzuki formula for higher orders can be recursively defined using a *fractal decomposition*. In particular, the fourth order formula $U_4(\Delta)$ can be constructed using the second-order one $U_2(\Delta)$ as follows.
$$
\begin{aligned}
U_2(\Delta) & = e^{-iA\Delta/2} e^{-iB\Delta} e^{-iA\Delta/2}; \\
U_4(\Delta) & = U_2(p\Delta)U_2(p\Delta)U_2((1 - 4p)\Delta)U_2(p\Delta)U_2(p\Delta); \\
p & = (4 - 4^{1/3})^{-1}.
\end{aligned}
$$

Because the Hamiltonian carries a `term_partition` populated from the
lattice's edge coloring, the Trotter builder consumes it directly: each
term group corresponds to one interaction type (e.g., ZZ coupling or X
field), and within each group the edges of each color form a
parallelizable layer with disjoint qubit supports. This tighter
grouping reduces the number of sequential exponentials per Trotter
step, and the improvement compounds through the Suzuki recursion.

In [ ]:
import math
from qdk_chemistry.algorithms.hamiltonian_unitary_builder.time_evolution.trotter import Trotter

t = 1.0   # total simulation time
dt = 0.5  # time per Trotter step

# The Trotter builder reads `hamiltonian.term_partition` automatically and
# uses the layered group structure to sort and reduce the schedule.
trotter = Trotter(order=4, num_divisions=math.ceil(t / dt), time=t)
evolution = trotter.run(hamiltonian)

container = evolution.get_container()
print(f"4th-order Trotter: {len(container.step_terms)} Pauli "
      f"exponentials per step, {container.step_reps} repetitions")
print(f"Qubits: {evolution.get_num_qubits()}")

## Building the circuit

The `TimeEvolutionUnitary` is converted to a `Circuit` using the
`PauliSequenceMapper`, which compiles the Pauli product formula into a
Q# circuit with a QIR representation.

In [ ]:
mapper = create("circuit_mapper", "pauli_sequence")
circuit = mapper.run(evolution)

print(f"Circuit has QIR: {circuit.qir is not None}")
print(f"Circuit has Q#:  {circuit.qsharp is not None}")

## Application wrapper

To obtain resource estimates we wrap the circuit in a QRE `Application`
instance. The `Circuit.get_qre_application()` method handles this
automatically, choosing the appropriate application type based on the
available circuit representation.

In [ ]:
app = circuit.get_qre_application()

## Majorana architectures

A QRE `Architecture` is a container for an Instruction Set Architecture (ISA): a list of instructions the hardware supports. The Majorana architecture supports single-qubit and two-qubit measurements in the X- and Z-bases, as well as a timing-based T-gate. The `Majorana` subclass provides the ISA for a user specified measurement error rate. Note the asymmetry in the assumed error rates for measurement/state preparation versus the unitary T-gate.

In [ ]:
from qdk.qre.models import Majorana

arch = Majorana(error_rate=1e-5)
arch

## Creating Application and ISA Queries

The core of the system is modeled through trace queries (from the top) and isa_queries (from the bottom).

The trace query applies layouts (`ISATransform` instances) to convert the operations from the application into logical operations supported by error correction codes and magic state factories. Here we expand fine rotation gates into T-gates using circuit synthesis. The `PSSPC` layout takes as an argument the options for the number of T-gates used per rotation. QRE will enumerate over this list and compute the error rate associated to each approximation, and the contribution to the overall error rate. The `LatticeSurgery` layout links the cat-state layout (which requires instruction MULTI_PAULI_MEAS) to the logical operation provided by the code (LATTICE_SURGERY). Its `slow_down_factor` option lets QRE deliberately stretch each logical operation in time, trading a longer runtime for far fewer magic-state factories (and hence fewer qubits); `slow_down_factor = 1` runs as fast as possible.

The ISA query provides the specific options for the code, in this case `ThreeAux` (an instance of the surface code). It provides the LATTICE_SURGERY instruction required by the trace. Magic states are not provided by the code, and so the `RoundBasedFactory` is a generic model that provides magic state distillation.

In [ ]:
from qdk.qre import estimate, plot_estimates, PSSPC, LatticeSurgery
from qdk.qre.models import ThreeAux, RoundBasedFactory
from qdk.qre.instruction_ids import LATTICE_SURGERY
from qdk.qre.property_keys import NUM_TS_PER_ROTATION, DISTANCE, PHYSICAL_COMPUTE_QUBITS

# Throttled sweep over rotation-synthesis precision (num_ts_per_rotation) and the
# lattice-surgery slow_down_factor.  We start the slow_down sweep at 3 (not 1)
# so the un-throttled, factory-heavy fast configurations -- which land at very
# high qubit counts (>200k) -- are never collected.  This keeps the estimate in
# the practical low-qubit regime, so no x-axis truncation is needed downstream.
trace_query = (
    app.q()
    * PSSPC.q(num_ts_per_rotation=[16, 17, 18, 19])
    * LatticeSurgery.q(slow_down_factor=[1.0 * j for j in range(3, 31)])
)

isa_query = (
    ThreeAux.q(distance=[11, 13, 15, 17, 19])
    * RoundBasedFactory.q(code_query=ThreeAux.q(distance=[5, 7, 11, 13, 15, 17, 19]))
)

results = estimate(app, arch, isa_query, trace_query, max_error=0.01, name="With term grouping")
results.add_column("compute_distance", lambda entry: entry.source[LATTICE_SURGERY].instruction[DISTANCE])
results.add_column("compute qubits", lambda entry: entry.properties[PHYSICAL_COMPUTE_QUBITS])
results.add_column("num_ts_per_rotation", lambda entry: entry.properties[NUM_TS_PER_ROTATION])
results.add_factory_summary_column()

results.as_frame()

## Does term grouping actually help?

The cells above used the lattice-aware **geometry-coloring** term partition
(`include_term_groups=True`, the default).  To quantify its benefit we now
repeat the *entire* pipeline — Trotter build, circuit mapping, and resource
estimation — with `include_term_groups=False`.

Without a partition the Trotter builder treats every Pauli term as its own
group, so commuting terms are no longer collapsed into parallel layers.  This
produces a longer product formula per step and therefore a more expensive
circuit.  Plotting both runs on the same axes shows the advantage of grouping.

In [ ]:
# Rebuild the SAME Ising Hamiltonian but WITHOUT the geometry-coloring partition.
hamiltonian_ungrouped = create_ising_hamiltonian(lattice, j=1.0, h=0.5, include_term_groups=False)
print(f"Ungrouped Hamiltonian term_partition: {hamiltonian_ungrouped.term_partition}")

# Trotter with no partition -> each Pauli term is its own group.
trotter_ungrouped = Trotter(order=4, num_divisions=math.ceil(t / dt), time=t)
evolution_ungrouped = trotter_ungrouped.run(hamiltonian_ungrouped)

container_ungrouped = evolution_ungrouped.get_container()
print(f"Ungrouped 4th-order Trotter: {len(container_ungrouped.step_terms)} Pauli "
      f"exponentials per step, {container_ungrouped.step_reps} repetitions")
print(f"Grouped (geometry coloring): {len(container.step_terms)} Pauli exponentials per step")

# Map to a circuit and wrap in a QRE application (same steps as the grouped run).
circuit_ungrouped = mapper.run(evolution_ungrouped)
app_ungrouped = circuit_ungrouped.get_qre_application()

# Resource estimation with the identical architecture, ISA, and slow_down_factor
# range as the grouped run, so the only difference between the two estimates is
# the presence of the term partition.
trace_query_ungrouped = (
    app_ungrouped.q()
    * PSSPC.q(num_ts_per_rotation=[16, 17, 18, 19])
    * LatticeSurgery.q(slow_down_factor=[1.0 * j for j in range(3, 31)])
)

results_ungrouped = estimate(
    app_ungrouped, arch, isa_query, trace_query_ungrouped,
    max_error=0.01, name="Without term grouping",
)
results_ungrouped.add_column("compute_distance", lambda entry: entry.source[LATTICE_SURGERY].instruction[DISTANCE])
results_ungrouped.add_column("compute qubits", lambda entry: entry.properties[PHYSICAL_COMPUTE_QUBITS])
results_ungrouped.add_column("num_ts_per_rotation", lambda entry: entry.properties[NUM_TS_PER_ROTATION])
results_ungrouped.add_factory_summary_column()

results_ungrouped.as_frame()

In [ ]:
fig = plot_estimates([results, results_ungrouped], runtime_unit="s", figsize=(15, 9.5),
                     scatter_args={"marker": "x", "s": 120, "linewidths": 2.5})
ax = fig.axes[0]
ax.set_xscale("linear")
ax.set_yscale("linear")
ax.set_xlabel(ax.get_xlabel(), fontsize=30)
ax.set_ylabel(ax.get_ylabel(), fontsize=30)
ax.tick_params(axis="both", labelsize=26)
ax.legend(fontsize=26, loc="upper right")
fig.set_layout_engine("constrained")
fig